In [9]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
from pathlib import Path

# Add project root to sys.path so we can import from src
# Assuming notebook is in project/notebooks/
project_root = Path('..').resolve()
sys.path.append(str(project_root))

# Import your actual project modules
from src.data_loader.cmapss.CMAPSSDataLoader import CMAPSSDataLoader
from src.data_loader.cmapss.CMAPSSTimeSeriesDataset import CMAPSSTimeSeriesDataset
from src.models.TransformerModel import TransformerModel
from src.models.LSTMModel import LSTMModel
from src.counterfactuals.core import BasisGenerator  
from src.counterfactuals.basis import BSplineBasis

# Device config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device("cpu")
print(f"Using device: {device}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using device: cpu


In [7]:
sys.path.append(os.path.dirname(os.path.abspath("/home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/src")))

In [ ]:
# Test the data loader
loader = CMAPSSDataLoader()
train_df, test_df, rul_true = loader.load_dataset('FD001')
print(f"\nTrain shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Add RUL to training data
train_df = loader.add_rul(train_df, max_rul=125)

# Add RUL to test data (for last cycle of each unit)
test_df_with_rul = test_df.copy()
max_cycles_test = test_df.groupby('unit_id')['cycle'].max().reset_index()
max_cycles_test['RUL'] = rul_true['RUL'].values
test_df_with_rul = test_df_with_rul.merge(max_cycles_test[['unit_id', 'RUL']], 
                                           on='unit_id', how='left')

# Remove constant features
train_df, test_df_with_rul, feature_cols = loader.remove_constant_features(
    train_df, test_df_with_rul, threshold=0.01
)

# Normalize data
train_norm, test_norm, mean_stats, std_stats = loader.normalize_data(
    train_df, test_df_with_rul, feature_cols
)

print(f"\nPreprocessed data shapes:")
print(f"Train: {train_norm.shape}")
print(f"Test: {test_norm.shape}")
print(f"Features: {len(feature_cols)}")

sequence_length = 50

train_dataset = CMAPSSTimeSeriesDataset(train_norm, sequence_length=sequence_length, 
                                  feature_cols=feature_cols)

# For test, we typically use the last sequence of each unit
test_sequences = []
test_targets = []
for unit_id in test_norm['unit_id'].unique():
    unit_data = test_norm[test_norm['unit_id'] == unit_id].sort_values('cycle')
    features = unit_data[feature_cols].values
    rul = unit_data['RUL'].values
    
    if len(features) >= sequence_length:
        test_sequences.append(features[-sequence_length:])
        test_targets.append(rul[-1])

test_sequences = np.array(test_sequences, dtype=np.float32)
test_targets = np.array(test_targets, dtype=np.float32)

print(f"\nTest sequences: {test_sequences.shape}")
print(f"Test targets: {test_targets.shape}")



Loaded FD001:
  Training samples: 20631, Units: 100
  Test samples: 13096, Units: 100

Train shape: (20631, 26)
Test shape: (13096, 26)
Removed 10 constant/low-variance features: ['setting_1', 'setting_2', 'setting_3', 'sensor_1', 'sensor_5', 'sensor_6', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Kept 14 features
Normalized 14 features

Preprocessed data shapes:
Train: (20631, 17)
Test: (13096, 17)
Features: 14
Created dataset with 15731 sequences
Sequence shape: (15731, 50, 14)
Feature dimension: 14

Test sequences: (93, 50, 14)
Test targets: (93,)


In [19]:
train_df.columns


Index(['unit_id', 'cycle', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7',
       'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13',
       'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21', 'RUL'],
      dtype='object')

In [20]:
test_df_with_rul.columns


Index(['unit_id', 'cycle', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7',
       'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13',
       'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21', 'RUL'],
      dtype='object')

In [23]:
# --- Initialize Model ---
# Must match your TransformerModel.py arguments
input_size = 14  # We know this from your output
model = TransformerModel(
    input_size=input_size,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.2
).to(device)

model_path = '../outputs/saved_models/Transformer_best.pth'
model_ready = False

# 1. Try Loading
if os.path.exists(model_path):
    try:
        #checkpoint = torch.load(model_path, map_location=device)
        # Handle key mismatch if saved with different variable names
        model.load_state_dict(torch.load(model_path, map_location=device), strict=False)
        print(f"✅ Loaded model from {model_path}")
        model_ready = True
    except Exception as e:
        print(f"⚠️ Checkpoint load failed: {e}")
        model_ready = False

model.eval()
#test model on test data
with torch.no_grad():
    test_inputs = torch.tensor(test_sequences, dtype=torch.float32).to(device)
    test_outputs = model(test_inputs)
    print(f"\nTest outputs shape: {test_outputs.shape}")

✅ Loaded model from ../outputs/saved_models/Transformer_best.pth

Test outputs shape: torch.Size([93, 1])


In [16]:
# Convert to Tensor (Shape: 93, 50, 14)
test_tensor = torch.tensor(np.array(test_sequences), dtype=torch.float32).to(device)

print(f"\nTest Tensor Shape: {test_tensor.shape}")


Test Tensor Shape: torch.Size([93, 50, 14])


In [ ]:
# Cell 1: Imports and Setup
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from pathlib import Path

# Add project root to sys.path
project_root = Path('..').resolve()
sys.path.append(str(project_root))

# Import project modules
from src.data_loader.cmapss.CMAPSSDataLoader import CMAPSSDataLoader
from src.data_loader.cmapss.CMAPSSTimeSeriesDataset import CMAPSSTimeSeriesDataset
from src.models.TransformerModel import TransformerModel
from src.models.LSTMModel import LSTMModel
from src.counterfactuals.core import BasisGenerator  
from src.counterfactuals.basis import BSplineBasis

# Device config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device("cpu")
print(f"Using device: {device}")

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Cell 2: Load and Preprocess Data
loader = CMAPSSDataLoader()
train_df, test_df, rul_true = loader.load_dataset('FD001')
print(f"\nTrain shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Add RUL to training data
train_df = loader.add_rul(train_df, max_rul=125)

# Add RUL to test data
test_df_with_rul = test_df.copy()
max_cycles_test = test_df.groupby('unit_id')['cycle'].max().reset_index()
max_cycles_test['RUL'] = rul_true['RUL'].values
test_df_with_rul = test_df_with_rul.merge(max_cycles_test[['unit_id', 'RUL']], 
                                           on='unit_id', how='left')

# Remove constant features
train_df, test_df_with_rul, feature_cols = loader.remove_constant_features(
    train_df, test_df_with_rul, threshold=0.01
)

# Normalize data
train_norm, test_norm, mean_stats, std_stats = loader.normalize_data(
    train_df, test_df_with_rul, feature_cols
)

print(f"\nPreprocessed data shapes:")
print(f"Train: {train_norm.shape}")
print(f"Test: {test_norm.shape}")
print(f"Features: {len(feature_cols)}")

# Cell 3: Load Trained Model
sequence_length = 50
input_size = len(feature_cols)

model = TransformerModel(
    input_size=input_size,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.2
).to(device)

model_path = '../outputs/saved_models/Transformer_best.pth'

if os.path.exists(model_path):
    try:
        model.load_state_dict(torch.load(model_path, map_location=device), strict=False)
        print(f"✅ Loaded model from {model_path}")
    except Exception as e:
        print(f"⚠️ Model load failed: {e}")
        raise

model.eval()
print("Model ready for inference")

# Cell 4: Helper Functions
def get_unit_sequences(df, unit_id, sequence_length, feature_cols):
    """Extract all sequences for a specific unit"""
    unit_data = df[df['unit_id'] == unit_id].sort_values('cycle')
    features = unit_data[feature_cols].values
    rul = unit_data['RUL'].values
    
    sequences = []
    ruls = []
    cycles = []
    
    for i in range(len(features) - sequence_length + 1):
        sequences.append(features[i:i+sequence_length])
        ruls.append(rul[i+sequence_length-1])
        cycles.append(unit_data['cycle'].iloc[i+sequence_length-1])
    
    return np.array(sequences), np.array(ruls), np.array(cycles)

def get_last_sequence(df, unit_id, sequence_length, feature_cols):
    """Get the last sequence for a unit (typical test scenario)"""
    unit_data = df[df['unit_id'] == unit_id].sort_values('cycle')
    features = unit_data[feature_cols].values
    rul = unit_data['RUL'].values[-1]
    
    if len(features) >= sequence_length:
        return features[-sequence_length:], rul
    else:
        return None, None

def predict_rul(model, sequence, device):
    """Predict RUL for a single sequence"""
    model.eval()
    with torch.no_grad():
        seq_tensor = torch.tensor(sequence, dtype=torch.float32).unsqueeze(0).to(device)
        pred = model(seq_tensor).item()
    return pred

# Cell 5: Generate Counterfactuals for Test Unit
# Select a test unit
test_unit_id = 24  # Change this to any test unit (1-100)

print(f"\n{'='*80}")
print(f"Generating Counterfactuals for Test Unit {test_unit_id}")
print(f"{'='*80}\n")

# Get the last sequence for this unit
query_seq, true_rul = get_last_sequence(test_norm, test_unit_id, sequence_length, feature_cols)

if query_seq is None:
    print(f"Unit {test_unit_id} has insufficient data")
else:
    # Convert to tensor
    query_instance = torch.tensor(query_seq, dtype=torch.float32).to(device)
    
    # Get current prediction
    current_pred = predict_rul(model, query_seq, device)
    
    print(f"Unit ID: {test_unit_id}")
    print(f"True RUL: {true_rul:.2f} cycles")
    print(f"Predicted RUL: {current_pred:.2f} cycles")
    print(f"Prediction Error: {abs(current_pred - true_rul):.2f} cycles")
    
    # Set target RUL (extend life by 30 cycles)
    target_rul = current_pred + 30.0
    print(f"\nTarget Counterfactual RUL: {target_rul:.2f} cycles")
    
    # Initialize Basis Generator
    generator = BasisGenerator(
        model=model,
        sequence_length=sequence_length,
        feature_dim=input_size,
        basis_type='bspline',
        num_basis=5,
        device=device
    )
    print("\n✅ Generator initialized")
    
    # Generate counterfactuals
    print("\nGenerating counterfactuals...")
    cfs = generator.generate(
        query_instance=query_instance,
        target_rul=target_rul,
        num_cfs=3,
        lr=0.05,
        max_iter=500,
        lambdas={
            'validity': 5.0,
            'prox': 0.1,
            'sparsity': 0.05,
            'diversity': 0.2
        },
        verbose=True
    )
    
    print(f"\n✅ Generated {len(cfs)} counterfactuals")
    
    # Verify predictions
    print("\nCounterfactual Predictions:")
    for i, cf in enumerate(cfs):
        cf_pred = predict_rul(model, cf.cpu().numpy(), device)
        print(f"  CF {i+1}: {cf_pred:.2f} cycles (Target: {target_rul:.2f})")

# Cell 6: Visualize Test Unit Counterfactuals
original_np = query_instance.cpu().numpy()
cfs_np = cfs.detach().cpu().numpy()

# Find top 3 most changed features
diff_magnitude = np.mean(np.abs(cfs_np - original_np), axis=(0,1))
top_indices = np.argsort(diff_magnitude)[-3:][::-1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feature_idx in enumerate(top_indices):
    ax = axes[i]
    fname = feature_cols[feature_idx]
    
    # Plot original
    ax.plot(original_np[:, feature_idx], 'k--', linewidth=2, label='Original')
    
    # Plot counterfactuals
    colors = ['#e74c3c', '#3498db', '#2ecc71']
    for j in range(len(cfs_np)):
        cf_pred = predict_rul(model, cfs[j].cpu().numpy(), device)
        ax.plot(cfs_np[j, :, feature_idx], color=colors[j], alpha=0.8, 
                label=f'CF {j+1} (RUL={cf_pred:.0f})')
    
    ax.set_title(f"Feature: {fname}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Time (Cycles)")
    if i == 0: 
        ax.set_ylabel("Normalized Sensor Value")
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.suptitle(f"Test Unit {test_unit_id}: Extending RUL from {current_pred:.0f} to {target_rul:.0f} cycles", 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'../outputs/counterfactuals_test_unit_{test_unit_id}.png', dpi=300, bbox_inches='tight')
plt.show()

# Cell 7: Generate Counterfactuals for Training Unit
train_unit_id = 42  # Change this to any train unit (1-100)

print(f"\n{'='*80}")
print(f"Generating Counterfactuals for Training Unit {train_unit_id}")
print(f"{'='*80}\n")

# Get sequences for this training unit
train_sequences, train_ruls, train_cycles = get_unit_sequences(
    train_norm, train_unit_id, sequence_length, feature_cols
)

print(f"Unit {train_unit_id} has {len(train_sequences)} sequences")

# Select a mid-life sequence (not too early, not too late)
mid_idx = len(train_sequences) // 2
query_seq_train = train_sequences[mid_idx]
true_rul_train = train_ruls[mid_idx]
cycle_num = train_cycles[mid_idx]

query_instance_train = torch.tensor(query_seq_train, dtype=torch.float32).to(device)

# Get current prediction
current_pred_train = predict_rul(model, query_seq_train, device)

print(f"Selected Sequence at Cycle: {cycle_num}")
print(f"True RUL: {true_rul_train:.2f} cycles")
print(f"Predicted RUL: {current_pred_train:.2f} cycles")

# Set target (reduce RUL by 20 cycles - simulate degradation)
target_rul_train = max(0, current_pred_train - 20.0)
print(f"\nTarget Counterfactual RUL: {target_rul_train:.2f} cycles")
print("(Simulating accelerated degradation)")

# Generate counterfactuals
print("\nGenerating counterfactuals...")
cfs_train = generator.generate(
    query_instance=query_instance_train,
    target_rul=target_rul_train,
    num_cfs=3,
    lr=0.05,
    max_iter=500,
    lambdas={
        'validity': 5.0,
        'prox': 0.1,
        'sparsity': 0.05,
        'diversity': 0.2
    },
    verbose=True
)

print(f"\n✅ Generated {len(cfs_train)} counterfactuals")

# Verify predictions
print("\nCounterfactual Predictions:")
for i, cf in enumerate(cfs_train):
    cf_pred = predict_rul(model, cf.cpu().numpy(), device)
    print(f"  CF {i+1}: {cf_pred:.2f} cycles (Target: {target_rul_train:.2f})")

# Cell 8: Visualize Training Unit Counterfactuals
original_train_np = query_instance_train.cpu().numpy()
cfs_train_np = cfs_train.detach().cpu().numpy()

# Find top 3 most changed features
diff_magnitude_train = np.mean(np.abs(cfs_train_np - original_train_np), axis=(0,1))
top_indices_train = np.argsort(diff_magnitude_train)[-3:][::-1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feature_idx in enumerate(top_indices_train):
    ax = axes[i]
    fname = feature_cols[feature_idx]
    
    # Plot original
    ax.plot(original_train_np[:, feature_idx], 'k--', linewidth=2, label='Original')
    
    # Plot counterfactuals
    colors = ['#e74c3c', '#3498db', '#2ecc71']
    for j in range(len(cfs_train_np)):
        cf_pred = predict_rul(model, cfs_train[j].cpu().numpy(), device)
        ax.plot(cfs_train_np[j, :, feature_idx], color=colors[j], alpha=0.8, 
                label=f'CF {j+1} (RUL={cf_pred:.0f})')
    
    ax.set_title(f"Feature: {fname}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Time (Cycles)")
    if i == 0: 
        ax.set_ylabel("Normalized Sensor Value")
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.suptitle(f"Train Unit {train_unit_id} (Cycle {cycle_num}): Reducing RUL from {current_pred_train:.0f} to {target_rul_train:.0f} cycles", 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'../outputs/counterfactuals_train_unit_{train_unit_id}.png', dpi=300, bbox_inches='tight')
plt.show()

# Cell 9: Compare All Features (Heatmap)
def plot_change_heatmap(original, cfs, feature_names, title):
    """Plot heatmap showing changes across all features"""
    original_np = original.cpu().numpy()
    cfs_np = cfs.detach().cpu().numpy()
    
    # Calculate average absolute change for each feature over time
    changes = []
    for cf in cfs_np:
        change = np.abs(cf - original_np)
        avg_change = np.mean(change, axis=0)  # Average over time
        changes.append(avg_change)
    
    changes = np.array(changes)
    
    plt.figure(figsize=(12, 6))
    sns.heatmap(changes, 
                xticklabels=feature_names,
                yticklabels=[f'CF {i+1}' for i in range(len(cfs_np))],
                cmap='YlOrRd',
                annot=True,
                fmt='.3f',
                cbar_kws={'label': 'Average Absolute Change'})
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Features')
    plt.ylabel('Counterfactuals')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Plot for test unit
plot_change_heatmap(query_instance, cfs, feature_cols, 
                    f"Test Unit {test_unit_id}: Feature Changes")

# Plot for training unit
plot_change_heatmap(query_instance_train, cfs_train, feature_cols, 
                    f"Train Unit {train_unit_id}: Feature Changes")

# Cell 10: Batch Processing - Multiple Units
def generate_cfs_for_multiple_units(df, unit_ids, model, generator, 
                                     sequence_length, feature_cols, 
                                     target_adjustment=30):
    """Generate counterfactuals for multiple units"""
    results = []
    
    for unit_id in unit_ids:
        query_seq, true_rul = get_last_sequence(df, unit_id, sequence_length, feature_cols)
        
        if query_seq is None:
            continue
        
        query_instance = torch.tensor(query_seq, dtype=torch.float32).to(device)
        current_pred = predict_rul(model, query_seq, device)
        target_rul = current_pred + target_adjustment
        
        try:
            cfs = generator.generate(
                query_instance=query_instance,
                target_rul=target_rul,
                num_cfs=1,  # Generate 1 CF per unit for efficiency
                lr=0.05,
                max_iter=300,
                lambdas={'validity': 5.0, 'prox': 0.1, 'sparsity': 0.05, 'diversity': 0.0},
                verbose=False
            )
            
            cf_pred = predict_rul(model, cfs[0].cpu().numpy(), device)
            
            results.append({
                'unit_id': unit_id,
                'true_rul': true_rul,
                'original_pred': current_pred,
                'target_rul': target_rul,
                'cf_pred': cf_pred,
                'success': abs(cf_pred - target_rul) < 5.0
            })
            
            print(f"Unit {unit_id}: Original={current_pred:.1f}, Target={target_rul:.1f}, CF={cf_pred:.1f}")
            
        except Exception as e:
            print(f"Failed for unit {unit_id}: {e}")
    
    return pd.DataFrame(results)

# Generate for first 10 test units
test_unit_ids = list(range(1, 11))
results_df = generate_cfs_for_multiple_units(
    test_norm, test_unit_ids, model, generator, 
    sequence_length, feature_cols, target_adjustment=30
)

print(f"\nSuccess Rate: {results_df['success'].mean()*100:.1f}%")
results_df

# Cell 11: Save Results
# Save counterfactuals to disk
output_dir = Path('../outputs/counterfactuals')
output_dir.mkdir(parents=True, exist_ok=True)

# Save as numpy arrays
np.save(output_dir / f'test_unit_{test_unit_id}_original.npy', original_np)
np.save(output_dir / f'test_unit_{test_unit_id}_cfs.npy', cfs_np)
np.save(output_dir / f'train_unit_{train_unit_id}_original.npy', original_train_np)
np.save(output_dir / f'train_unit_{train_unit_id}_cfs.npy', cfs_train_np)

# Save metadata
metadata = {
    'test_unit': {
        'unit_id': test_unit_id,
        'true_rul': float(true_rul),
        'original_pred': float(current_pred),
        'target_rul': float(target_rul)
    },
    'train_unit': {
        'unit_id': train_unit_id,
        'cycle': int(cycle_num),
        'true_rul': float(true_rul_train),
        'original_pred': float(current_pred_train),
        'target_rul': float(target_rul_train)
    },
    'feature_cols': feature_cols
}

import json
with open(output_dir / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Results saved to {output_dir}")